# COVFIL / BOXER Format Round-Trip Tests

Tests using actual NJOY output files for Fe-56 at 293.6 K.

**Data files** (`260560.02`):
- `.gendf` — COVFIL (ENDF-style text)
- `.boxer` — BOXER card-image (ASCII)

**Tests:**
1. Read both files and compare metadata + matrices
2. COVFIL → write BOXER → read BOXER → compare
3. BOXER self-roundtrip (write → read)
4. Cross-format full cycle: COVFIL → BOXER → COVFIL → compare
5. Inspect written BOXER file structure
6. COVFIL self-roundtrip (write → read)
7. Compression/decompression unit tests

In [ ]:
import tempfile
import os
import numpy as np

from kika.cov import read_covfil, write_covfil, read_boxer, write_boxer
from kika.cov.cross_section_covariance import CrossSectionCovariance
from kika.cov.parse_covmat import (
    _compress_boxer, _decompress_boxer,
    _format_boxer_header, _parse_boxer_header,
    _choose_boxer_ncf,
)

DATA_DIR = r'C:\Users\Usuario\Documents\Archivos temporales\cov\unassigned\293.6'
COVFIL_FILE = os.path.join(DATA_DIR, '260560.02.gendf')
BOXER_FILE  = os.path.join(DATA_DIR, '260560.02.boxer')

assert os.path.isfile(COVFIL_FILE), f'Missing: {COVFIL_FILE}'
assert os.path.isfile(BOXER_FILE),  f'Missing: {BOXER_FILE}'
print('Data files found.')

## Test 1 — Read Both Files and Compare

Read the COVFIL and BOXER files independently and compare what each format preserves.

In [ ]:
# Read both files
cov_from_covfil = read_covfil(COVFIL_FILE)
cov_from_boxer  = read_boxer(BOXER_FILE)

print('=== COVFIL ===')
print(f'  type:          {type(cov_from_covfil).__name__}')
print(f'  num_groups:    {cov_from_covfil.num_groups}')
print(f'  num_matrices:  {cov_from_covfil.num_matrices}')
print(f'  energy_grid:   {len(cov_from_covfil.energy_grid) if cov_from_covfil.energy_grid else "None"} boundaries')
print(f'  cross_sections:{len(cov_from_covfil.cross_sections)} entries')
print(f'  energy_unit:   {cov_from_covfil.energy_unit}')
print(f'  reactions:     {list(zip(cov_from_covfil.reaction_rows, cov_from_covfil.reaction_cols))}')
print(f'  isotopes:      {list(zip(cov_from_covfil.isotope_rows, cov_from_covfil.isotope_cols))}')

print()
print('=== BOXER ===')
print(f'  type:          {type(cov_from_boxer).__name__}')
print(f'  num_groups:    {cov_from_boxer.num_groups}')
print(f'  num_matrices:  {cov_from_boxer.num_matrices}')
print(f'  energy_grid:   {len(cov_from_boxer.energy_grid) if cov_from_boxer.energy_grid else "None"} boundaries')
print(f'  cross_sections:{len(cov_from_boxer.cross_sections)} entries')
print(f'  energy_unit:   {cov_from_boxer.energy_unit}')
print(f'  reactions:     {list(zip(cov_from_boxer.reaction_rows, cov_from_boxer.reaction_cols))}')
print(f'  isotopes:      {list(zip(cov_from_boxer.isotope_rows, cov_from_boxer.isotope_cols))}')

print()
print('=== COMPARISON ===')
groups_match = cov_from_covfil.num_groups == cov_from_boxer.num_groups
nmat_match   = cov_from_covfil.num_matrices == cov_from_boxer.num_matrices
print(f'  num_groups match:   {groups_match} ({cov_from_covfil.num_groups} vs {cov_from_boxer.num_groups})')
print(f'  num_matrices match: {nmat_match} ({cov_from_covfil.num_matrices} vs {cov_from_boxer.num_matrices})')

covfil_pairs = list(zip(cov_from_covfil.isotope_rows, cov_from_covfil.reaction_rows,
                        cov_from_covfil.isotope_cols, cov_from_covfil.reaction_cols))
boxer_pairs  = list(zip(cov_from_boxer.isotope_rows, cov_from_boxer.reaction_rows,
                        cov_from_boxer.isotope_cols, cov_from_boxer.reaction_cols))
print(f'  reaction pairs match: {set(covfil_pairs) == set(boxer_pairs)}')
if set(covfil_pairs) != set(boxer_pairs):
    print(f'    Only in COVFIL: {set(covfil_pairs) - set(boxer_pairs)}')
    print(f'    Only in BOXER:  {set(boxer_pairs) - set(covfil_pairs)}')

if cov_from_covfil.energy_grid and cov_from_boxer.energy_grid:
    eg_arr_c = np.array(cov_from_covfil.energy_grid)
    eg_arr_b = np.array(cov_from_boxer.energy_grid)
    if len(eg_arr_c) == len(eg_arr_b):
        eg_max_rel = np.max(np.abs(
            (eg_arr_c - eg_arr_b) / np.where(eg_arr_c != 0, eg_arr_c, 1)
        ))
        print(f'  energy_grid close (rtol=1e-4): {np.allclose(eg_arr_c, eg_arr_b, rtol=1e-4)} (max_rel_diff={eg_max_rel:.2e})')
    else:
        print(f'  energy_grid length mismatch: {len(eg_arr_c)} vs {len(eg_arr_b)}')

covfil_xs_keys = set(cov_from_covfil.cross_sections.keys())
boxer_xs_keys  = set(cov_from_boxer.cross_sections.keys())
print(f'  XS keys in COVFIL: {sorted(covfil_xs_keys)}')
print(f'  XS keys in BOXER:  {sorted(boxer_xs_keys)}')
if covfil_xs_keys != boxer_xs_keys:
    print(f'    Only in COVFIL: {covfil_xs_keys - boxer_xs_keys}')
    print(f'    Only in BOXER:  {boxer_xs_keys - covfil_xs_keys}')

# Build lookup for boxer matrices by reaction pair
boxer_lookup = {}
for i, pair in enumerate(boxer_pairs):
    boxer_lookup[pair] = i

print()
print('  Matrix comparison (matching by reaction pair):')
for i in range(cov_from_covfil.num_matrices):
    pair = covfil_pairs[i]
    if pair in boxer_lookup:
        j = boxer_lookup[pair]
        orig = cov_from_covfil.matrices[i]
        comp = cov_from_boxer.matrices[j]
        if orig.shape != comp.shape:
            print(f'    [{i}] MT=({pair[1]},{pair[3]}): SHAPE MISMATCH {orig.shape} vs {comp.shape}')
            continue
        abs_diff = np.max(np.abs(orig - comp))
        mask = np.abs(orig) > 1e-30
        rel_diff = np.max(np.abs((orig[mask] - comp[mask]) / orig[mask])) if mask.any() else 0.0
        print(f'    [{i}] MT=({pair[1]},{pair[3]}): max_abs={abs_diff:.2e}, max_rel={rel_diff:.2e}')
    else:
        print(f'    [{i}] MT=({pair[1]},{pair[3]}): NOT FOUND in BOXER')

## Test 2 — COVFIL → write BOXER → read BOXER (covfil-origin roundtrip)

In [ ]:
# Start from COVFIL, write to BOXER, read back
mf33 = read_covfil(COVFIL_FILE)

tmp_boxer = tempfile.mktemp(suffix='.boxer')
write_boxer(mf33, tmp_boxer, hlibid='TST', hdescr='covfil-origin roundtrip', nvf=10)
mf33_rt = read_boxer(tmp_boxer)

print(f'Original (COVFIL): {mf33.num_matrices} matrices, {mf33.num_groups} groups, '
      f'{len(mf33.cross_sections)} XS')
print(f'Round-trip (BOXER): {mf33_rt.num_matrices} matrices, {mf33_rt.num_groups} groups, '
      f'{len(mf33_rt.cross_sections)} XS')

assert mf33_rt.num_groups == mf33.num_groups
assert mf33_rt.num_matrices == mf33.num_matrices
assert np.allclose(mf33.energy_grid, mf33_rt.energy_grid, rtol=1e-3)
print('Energy grid: MATCH')

for i in range(mf33.num_matrices):
    mt_r, mt_c = mf33.reaction_rows[i], mf33.reaction_cols[i]
    orig, rt = mf33.matrices[i], mf33_rt.matrices[i]
    mask = np.abs(orig) > 1e-30
    rel_diff = np.max(np.abs((orig[mask] - rt[mask]) / orig[mask])) if mask.any() else 0.0
    abs_diff = np.max(np.abs(orig - rt))
    status = 'OK' if (rel_diff < 1e-2 or abs_diff < 1e-6) else 'FAIL'
    print(f'  MT=({mt_r},{mt_c}): max_rel={rel_diff:.2e}, max_abs={abs_diff:.2e} [{status}]')
    assert rel_diff < 1e-2 or abs_diff < 1e-6

for key in mf33.cross_sections:
    assert key in mf33_rt.cross_sections, f'Missing XS key: {key}'
    assert np.allclose(mf33.cross_sections[key], mf33_rt.cross_sections[key], rtol=1e-3)
print(f'Cross sections ({len(mf33.cross_sections)} entries): MATCH')

os.unlink(tmp_boxer)
print('\nTest 2 PASSED.')

## Test 3 — BOXER → write COVFIL → read COVFIL (boxer-origin roundtrip)

In [ ]:
# Start from BOXER, write to COVFIL, read back
boxer_orig = read_boxer(BOXER_FILE)

tmp_covfil = tempfile.mktemp(suffix='.gendf')
write_covfil(boxer_orig, tmp_covfil, tape_label='boxer-origin roundtrip', temperature=293.6)
boxer_rt = read_covfil(tmp_covfil)

print(f'Original (BOXER):    {boxer_orig.num_matrices} matrices, {boxer_orig.num_groups} groups, '
      f'{len(boxer_orig.cross_sections)} XS')
print(f'Round-trip (COVFIL): {boxer_rt.num_matrices} matrices, {boxer_rt.num_groups} groups, '
      f'{len(boxer_rt.cross_sections)} XS')

assert boxer_rt.num_groups == boxer_orig.num_groups
assert boxer_rt.num_matrices == boxer_orig.num_matrices

# COVFIL is lossless, so boxer->covfil->read should be near-exact
for i in range(boxer_orig.num_matrices):
    mt_r, mt_c = boxer_orig.reaction_rows[i], boxer_orig.reaction_cols[i]
    orig, rt = boxer_orig.matrices[i], boxer_rt.matrices[i]
    mask = np.abs(orig) > 1e-30
    rel_diff = np.max(np.abs((orig[mask] - rt[mask]) / orig[mask])) if mask.any() else 0.0
    abs_diff = np.max(np.abs(orig - rt))
    status = 'OK' if (rel_diff < 1e-4 or abs_diff < 1e-10) else 'WARN'
    print(f'  MT=({mt_r},{mt_c}): max_rel={rel_diff:.2e}, max_abs={abs_diff:.2e} [{status}]')

os.unlink(tmp_covfil)
print('\nTest 3 PASSED.')

## Test 4 — BOXER self-roundtrip (read BOXER → write BOXER → read BOXER)

In [ ]:
# Read BOXER, write BOXER, read back — tests our own write/read consistency
boxer_orig = read_boxer(BOXER_FILE)

tmp_boxer = tempfile.mktemp(suffix='.boxer')
write_boxer(boxer_orig, tmp_boxer, hlibid='SLF', hdescr='self-roundtrip', nvf=10)
boxer_rt = read_boxer(tmp_boxer)

print(f'Original:   {boxer_orig.num_matrices} matrices, {boxer_orig.num_groups} groups')
print(f'Round-trip: {boxer_rt.num_matrices} matrices, {boxer_rt.num_groups} groups')

assert boxer_rt.num_groups == boxer_orig.num_groups
assert boxer_rt.num_matrices == boxer_orig.num_matrices
assert np.allclose(boxer_orig.energy_grid, boxer_rt.energy_grid, rtol=1e-3)

for i in range(boxer_orig.num_matrices):
    mt_r, mt_c = boxer_orig.reaction_rows[i], boxer_orig.reaction_cols[i]
    orig, rt = boxer_orig.matrices[i], boxer_rt.matrices[i]
    mask = np.abs(orig) > 1e-30
    rel_diff = np.max(np.abs((orig[mask] - rt[mask]) / orig[mask])) if mask.any() else 0.0
    abs_diff = np.max(np.abs(orig - rt))
    status = 'OK' if (rel_diff < 1e-2 or abs_diff < 1e-6) else 'FAIL'
    print(f'  MT=({mt_r},{mt_c}): max_rel={rel_diff:.2e}, max_abs={abs_diff:.2e} [{status}]')
    assert rel_diff < 1e-2 or abs_diff < 1e-6

os.unlink(tmp_boxer)
print('\nTest 4 PASSED.')

## Test 5 — Full cross-format cycle: COVFIL → BOXER → COVFIL → compare with original

In [ ]:
# Full cycle: covfil -> boxer -> covfil -> compare with original covfil
mf33 = read_covfil(COVFIL_FILE)

tmp_boxer = tempfile.mktemp(suffix='.boxer')
write_boxer(mf33, tmp_boxer, nvf=10)
mf33_boxer = read_boxer(tmp_boxer)

tmp_covfil = tempfile.mktemp(suffix='.gendf')
write_covfil(mf33_boxer, tmp_covfil, tape_label='cross-format cycle')
mf33_final = read_covfil(tmp_covfil)

print(f'Original (COVFIL):  {mf33.num_matrices} matrices, {mf33.num_groups} groups')
print(f'Final (COVFIL):     {mf33_final.num_matrices} matrices, {mf33_final.num_groups} groups')

assert mf33_final.num_matrices == mf33.num_matrices
assert mf33_final.num_groups == mf33.num_groups

for i in range(mf33.num_matrices):
    orig, final = mf33.matrices[i], mf33_final.matrices[i]
    mask = np.abs(orig) > 1e-30
    rel_diff = np.max(np.abs((orig[mask] - final[mask]) / orig[mask])) if mask.any() else 0.0
    abs_diff = np.max(np.abs(orig - final))
    mt_r, mt_c = mf33.reaction_rows[i], mf33.reaction_cols[i]
    status = 'OK' if (rel_diff < 1e-2 or abs_diff < 1e-6) else 'FAIL'
    print(f'  MT=({mt_r},{mt_c}): max_rel={rel_diff:.2e}, max_abs={abs_diff:.2e} [{status}]')
    assert rel_diff < 1e-2 or abs_diff < 1e-6

os.unlink(tmp_boxer)
os.unlink(tmp_covfil)
print('\nTest 5 PASSED.')

## Test 6 — Inspect written BOXER file structure and compare with original BOXER

In [ ]:
# Compare structure of original BOXER file vs our written BOXER
import math
from kika.cov.parse_covmat import _BOXER_VALUE_FORMATS, _BOXER_CONTROL_FORMATS

def extract_boxer_headers(filepath):
    """Extract all block headers from a BOXER file."""
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    headers = []
    cursor = 0
    while cursor < len(lines):
        hdr = _parse_boxer_header(lines[cursor])
        hdr['line_num'] = cursor + 1
        headers.append(hdr)
        cursor += 1
        if hdr['nval'] > 0:
            vpl, _ = _BOXER_VALUE_FORMATS[hdr['nvf']]
            cursor += math.ceil(hdr['nval'] / vpl)
        if hdr['ncon'] > 0:
            vpl, _ = _BOXER_CONTROL_FORMATS[hdr['ncf']]
            cursor += math.ceil(hdr['ncon'] / vpl)
    return headers, lines

# Original BOXER
orig_hdrs, orig_lines = extract_boxer_headers(BOXER_FILE)

# Our written BOXER
mf33 = read_covfil(COVFIL_FILE)
tmp_boxer = tempfile.mktemp(suffix='.boxer')
write_boxer(mf33, tmp_boxer, hlibid='TST', hdescr='structure comparison', nvf=10)
written_hdrs, written_lines = extract_boxer_headers(tmp_boxer)

itype_names = {0: 'ENERGY', 1: 'XS', 2: 'STD-DEV', 3: 'COV', 4: 'CORR'}

print(f'Original BOXER: {len(orig_lines)} lines, {len(orig_hdrs)} blocks')
print(f'Written BOXER:  {len(written_lines)} lines, {len(written_hdrs)} blocks')
print()

print('=== Original BOXER block headers ===')
for h in orig_hdrs:
    print(f'  Line {h["line_num"]:4d}: ITYPE={h["itype"]} ({itype_names.get(h["itype"], "?"):8s}) '
          f'MAT={h["mat"]:5d} MT={h["mt"]:3d} MAT1={h["mat1"]:5d} MT1={h["mt1"]:3d} '
          f'NVAL={h["nval"]:4d} NROWH={h["nrowh"]:3d} NCOLH={h["ncolh"]:3d}')

print()
print('=== Written BOXER block headers ===')
for h in written_hdrs:
    print(f'  Line {h["line_num"]:4d}: ITYPE={h["itype"]} ({itype_names.get(h["itype"], "?"):8s}) '
          f'MAT={h["mat"]:5d} MT={h["mt"]:3d} MAT1={h["mat1"]:5d} MT1={h["mt1"]:3d} '
          f'NVAL={h["nval"]:4d} NROWH={h["nrowh"]:3d} NCOLH={h["ncolh"]:3d}')

# Compare block counts by type
from collections import Counter
orig_types = Counter(h['itype'] for h in orig_hdrs)
written_types = Counter(h['itype'] for h in written_hdrs)
print()
print('Block type counts:')
for t in sorted(set(list(orig_types.keys()) + list(written_types.keys()))):
    name = itype_names.get(t, '?')
    print(f'  ITYPE={t} ({name:8s}): orig={orig_types.get(t,0)}, written={written_types.get(t,0)}')

os.unlink(tmp_boxer)
print('\nTest 6 done.')

## Test 7 — Compression / Decompression Unit Tests

In [ ]:
# --- 1a: Identity matrix (symmetric) ---
n = 10
eye = np.eye(n)
xval, icons = _compress_boxer(eye, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(eye, result), f'Identity mismatch: max_diff={np.max(np.abs(eye - result)):.2e}'
print(f'7a Identity ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1b: Random symmetric matrix ---
np.random.seed(42)
n = 20
A = np.random.randn(n, n)
A = (A + A.T) / 2
xval, icons = _compress_boxer(A, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(A, result), f'Random symmetric mismatch: max_diff={np.max(np.abs(A - result)):.2e}'
print(f'7b Random symmetric ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1c: Sparse symmetric matrix ---
n = 15
S = np.zeros((n, n))
S[3, 3] = 1.5
S[3, 5] = 0.3; S[5, 3] = 0.3
S[5, 5] = 2.0
S[10, 10] = 0.7
xval, icons = _compress_boxer(S, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(S, result), f'Sparse mismatch: max_diff={np.max(np.abs(S - result)):.2e}'
print(f'7c Sparse symmetric ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1d: Non-symmetric matrix ---
n = 8
B = np.random.randn(n, n)
xval, icons = _compress_boxer(B, symmetric=False)
result = _decompress_boxer(xval, icons, n, n)
assert np.allclose(B, result), f'Non-symmetric mismatch: max_diff={np.max(np.abs(B - result)):.2e}'
print(f'7d Non-symmetric ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1e: Zero matrix ---
n = 5
Z = np.zeros((n, n))
xval, icons = _compress_boxer(Z, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(Z, result), 'Zero matrix mismatch'
print(f'7e Zero matrix ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

# --- 1f: Matrix with carry-down pattern ---
n = 6
C = np.ones((n, n)) * 3.14
xval, icons = _compress_boxer(C, symmetric=True)
result = _decompress_boxer(xval, icons, n, 0)
assert np.allclose(C, result), 'Carry-down mismatch'
print(f'7f Carry-down ({n}x{n}): OK  (nval={len(xval)}, ncon={len(icons)})')

print()
print('All compression/decompression unit tests passed.')